<a href="https://colab.research.google.com/github/arvinmungara-alt/Agentic-AI/blob/main/Agentic_AI_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-google-genai duckduckgo-search


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key="YOUR_KEY")


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

from langchain_core.tools import Tool

search_tool = DuckDuckGoSearchRun()
tools = [
    Tool(
        name="WebSearch",
        func=search_tool.run,
        description="Useful for current events"
    )
]


In [ ]:
# 1. Define the query
query = "There are reports of an unusual respiratory illness in a specific zip code. Investigate and summarize."

# 2. Bind the tools directly to your Gemini LLM (no complex agent modules required!)
# This lets Gemini know it has the power to use your WebSearch tool
llm_with_tools = llm.bind_tools(tools)

print("Starting Agentic Investigation...\n")

# 3. Tell the model to think about what to do
initial_response = llm_with_tools.invoke(query)

# Check if Gemini decided it needs to use a tool
if initial_response.tool_calls:
    print("🤖 Agent decided to take Action: Using WebSearch tool...")

    # Extract the tool arguments that Gemini generated
    tool_call = initial_response.tool_calls[0]
    tool_query = tool_call["args"].get("query", query)

    # Run the tool (DuckDuckGo Search)
    print(f"🔍 Searching for: '{tool_query}'")
    search_results = search_tool.run(tool_query)

    print("\n📬 Observation received from tool. Synthesizing final report...\n")

    # Feed the search results back to Gemini to get the final summary
    final_prompt = f"""
    The user asked: {query}

    We ran a web search and found this raw data:
    {search_results}

    Please synthesize this information into a clear public health triage summary.
    """
    final_response = llm.invoke(final_prompt)
    response_text = final_response.content
else:
    # If it didn't need the tool, just take its direct answer
    response_text = initial_response.content

# 4. Print the final result
print("--- Final Agent Summary ---")
print(response_text)

Starting Agentic Investigation...



ChatGoogleGenerativeAIError: Error calling model 'gemini-pro' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}